# 🎙️ XTTS v2 Vietnamese Fine-Tuning

**Datasets cần add vào notebook này:**
| Dataset | Dùng để |
|---|---|
| `tinthnhphm21022004/data-speech-to-text` | File audio WAV |
| `thanhphamtien2102224/weight-phowhisper` | Manifest JSONL (train + test) |
| `xtts-pipeline` *(upload code)* | Code pipeline |

**Settings:**
- Accelerator → **GPU T4 x1**
- Internet → **ON**

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1 — Kiểm tra GPU + đường dẫn dataset
# ════════════════════════════════════════════════════════════
import os, torch

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

# ── Đường dẫn thực tế từ Kaggle datasets ──────────────────────
AUDIO_ROOT    = '/kaggle/input/datasets/tinthnhphm21022004/data-speech-to-text'
MANIFEST_ROOT = '/kaggle/input/datasets/thanhphamtien2102224/weight-phowhisper'
TRAIN_JSONL   = f'{MANIFEST_ROOT}/train_full_manifest.jsonl'
TEST_JSONL    = f'{MANIFEST_ROOT}/test_manifest.jsonl'

# Kiểm tra tồn tại
for p in [AUDIO_ROOT, TRAIN_JSONL, TEST_JSONL]:
    status = '✅' if os.path.exists(p) else '❌ KHÔNG TÌM THẤY'
    print(f'{status}  {p}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 2 — Xem thử vài dòng manifest để biết format
# ════════════════════════════════════════════════════════════
import json

print('=== 3 dòng đầu train_full_manifest.jsonl ===')
with open(TRAIN_JSONL, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 3: break
        item = json.loads(line.strip())
        print(json.dumps(item, ensure_ascii=False, indent=2))
        print()

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 3 — Cài thư viện
# ════════════════════════════════════════════════════════════
!pip install -q TTS==0.22.0
!pip install -q huggingface_hub librosa soundfile
# !pip install -q peft   # bỏ comment nếu muốn dùng LoRA
print('✅ Cài đặt xong')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 4 — Load code pipeline
# ════════════════════════════════════════════════════════════
import sys, shutil

# Nếu upload code dưới dạng Kaggle Dataset tên 'xtts-pipeline'
CODE_SRC = '/kaggle/input/xtts-pipeline/xtts_finetune'
CODE_DST = '/kaggle/working/xtts_finetune'

if os.path.isdir(CODE_SRC):
    if not os.path.isdir(CODE_DST):
        shutil.copytree(CODE_SRC, CODE_DST)
    sys.path.insert(0, '/kaggle/working')
    print(f'✅ Code loaded từ: {CODE_SRC}')
else:
    print(f'⚠️  Không tìm thấy {CODE_SRC}')
    print('   → Hãy upload folder xtts_finetune/ lên Kaggle Datasets với tên xtts-pipeline')

# Test import
from xtts_finetune.config import TrainingConfig
from xtts_finetune.utils import get_logger, set_seed
print('✅ Import thành công')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 5 — Tìm file reference.wav trong audio dataset
# ════════════════════════════════════════════════════════════

# Tìm 1 file wav bất kỳ để dùng làm reference (speaker conditioning)
# Lý tưởng nhất: 1 file 5-10 giây, giọng rõ, không nhiễu
REFERENCE_WAV = None

# Thử tìm file tên reference.wav trước
for root, dirs, files in os.walk(AUDIO_ROOT):
    for f in files:
        if f.lower() == 'reference.wav':
            REFERENCE_WAV = os.path.join(root, f)
            break
    if REFERENCE_WAV:
        break

# Nếu không có, lấy file wav đầu tiên tìm được
if not REFERENCE_WAV:
    for root, dirs, files in os.walk(AUDIO_ROOT):
        for f in sorted(files):
            if f.lower().endswith('.wav'):
                REFERENCE_WAV = os.path.join(root, f)
                break
        if REFERENCE_WAV:
            break

if REFERENCE_WAV:
    print(f'✅ Reference audio: {REFERENCE_WAV}')
    # Kiểm tra duration
    import torchaudio
    info = torchaudio.info(REFERENCE_WAV)
    dur  = info.num_frames / info.sample_rate
    print(f'   Duration: {dur:.2f}s | SR: {info.sample_rate} Hz')
    if dur < 3:
        print('   ⚠️  File quá ngắn, nên dùng file 5-10 giây')
else:
    print('❌ Không tìm thấy file WAV nào trong', AUDIO_ROOT)

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 6 — Cấu hình training
# ════════════════════════════════════════════════════════════
from xtts_finetune.config import TrainingConfig

WORKING    = '/kaggle/working'
OUTPUT_DIR = f'{WORKING}/output'

config = TrainingConfig(
    # ── Model ──────────────────────────────────────────────
    hf_repo_id     = 'anhnh2002/vnTTS',
    base_model_dir = f'{WORKING}/base_model',

    # ── Data ───────────────────────────────────────────────
    # Manifest JSONL từ dataset weight-phowhisper
    train_manifest = TRAIN_JSONL,
    val_manifest   = TEST_JSONL,

    # Reference audio cho speaker conditioning
    reference_audio = REFERENCE_WAV,

    # ⚠️  QUAN TRỌNG: Remap đường dẫn audio trong manifest
    # Manifest của bạn có thể chứa path cũ như:
    #   /kaggle/input/data-speech-to-text/...
    # Cần map sang path thực tế trên Kaggle:
    #   /kaggle/input/datasets/tinthnhphm21022004/data-speech-to-text/...
    audio_root_remap = {
        '/kaggle/input/data-speech-to-text': AUDIO_ROOT,
        # Thêm các prefix cũ khác nếu cần:
        # '/old/path/prefix': '/new/path/prefix',
    },

    patch_size = 5000,

    # ── Output ─────────────────────────────────────────────
    output_dir     = OUTPUT_DIR,
    checkpoint_dir = f'{OUTPUT_DIR}/checkpoints',
    sample_dir     = f'{OUTPUT_DIR}/samples',
    log_dir        = f'{OUTPUT_DIR}/logs',

    # ── Training (T4 16GB optimized) ───────────────────────
    batch_size          = 2,
    grad_accum_steps    = 8,     # effective batch = 16
    learning_rate       = 2e-5,
    epochs_per_patch    = 1,

    # ── Memory ─────────────────────────────────────────────
    use_fp16               = True,
    gradient_checkpointing = True,
    freeze_encoder         = True,

    # ── Speaker ────────────────────────────────────────────
    speaker_mode = 'single',

    # ── Kaggle ─────────────────────────────────────────────
    zip_checkpoints = True,
    num_workers     = 2,
    seed            = 42,
)

print('✅ Config sẵn sàng')
print(f'   Train manifest : {config.train_manifest}')
print(f'   Val manifest   : {config.val_manifest}')
print(f'   Audio remap    : {config.audio_root_remap}')
print(f'   Reference wav  : {config.reference_audio}')
print(f'   Effective batch: {config.batch_size * config.grad_accum_steps}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 7 — Kiểm tra manifest + path remap trước khi train
# ════════════════════════════════════════════════════════════
from xtts_finetune.dataset import load_manifest, validate_and_filter

logger = get_logger('kaggle', config.log_dir)
set_seed(42)

print('📂 Loading train manifest...')
train_raw = load_manifest(config.train_manifest, logger,
                          audio_root_remap=config.audio_root_remap)

print('\n📂 Loading val/test manifest...')
val_raw = load_manifest(config.val_manifest, logger,
                        audio_root_remap=config.audio_root_remap)

# Kiểm tra 3 mẫu đầu xem path có đúng không
print('\n🔍 Kiểm tra 3 mẫu đầu (sau remap):')
for s in train_raw[:3]:
    exists = '✅' if os.path.isfile(s['audio']) else '❌'
    print(f'  {exists} {s["audio"]}')
    print(f'     text: "{s["text"][:80]}"')

# Đếm file tồn tại trong 100 mẫu đầu
sample_check = train_raw[:100]
found = sum(1 for s in sample_check if os.path.isfile(s['audio']))
print(f'\n📊 Kiểm tra 100 mẫu đầu: {found}/100 file tồn tại')
if found < 90:
    print('⚠️  Nhiều file không tìm thấy! Kiểm tra lại audio_root_remap trong Cell 6')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 8 — Validate + filter dataset
# ════════════════════════════════════════════════════════════
print('🔍 Validating train samples...')
train_samples = validate_and_filter(train_raw, config, logger)

print('\n🔍 Validating val samples...')
val_samples = validate_and_filter(val_raw, config, logger)

print(f'\n✅ Train: {len(train_samples):,} mẫu hợp lệ')
print(f'✅ Val  : {len(val_samples):,} mẫu hợp lệ')

if len(train_samples) == 0:
    raise RuntimeError('❌ Không có mẫu train hợp lệ! Kiểm tra lại audio_root_remap')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 9 — Download base model XTTS từ HuggingFace
# ════════════════════════════════════════════════════════════
from xtts_finetune.model_loader import download_base_model

download_base_model(config, logger)
print('✅ Base model sẵn sàng')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 10 — Load model + cấu hình
# ════════════════════════════════════════════════════════════
from xtts_finetune.model_loader import (
    load_xtts_model,
    configure_trainable_params,
    enable_gradient_checkpointing,
    extract_speaker_embedding,
)
from xtts_finetune.utils import log_gpu_memory

# Load XTTS
model, xtts_config = load_xtts_model(config, logger)

# Freeze encoder, chỉ train decoder + speaker layers
model = configure_trainable_params(model, config, logger)

# Gradient checkpointing tiết kiệm VRAM
enable_gradient_checkpointing(model, logger)

log_gpu_memory(logger, 'sau khi load model')

# Extract speaker embedding từ reference audio
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
speaker_embedding = extract_speaker_embedding(
    model, xtts_config, config.reference_audio, device, logger
)

if speaker_embedding is not None:
    print(f'✅ Speaker embedding shape: {speaker_embedding.shape}')
else:
    print('⚠️  Không extract được speaker embedding, dùng default')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 11 — BẮT ĐẦU TRAINING
# ════════════════════════════════════════════════════════════
from xtts_finetune.trainer import XTTSTrainer

trainer = XTTSTrainer(
    model            = model,
    xtts_config      = xtts_config,
    config           = config,
    speaker_embedding= speaker_embedding,
)

# ── RESUME (nếu session bị ngắt) ──────────────────────────────
# Bỏ comment nếu muốn resume:
# from xtts_finetune.utils import find_latest_checkpoint
# from xtts_finetune.model_loader import load_checkpoint
# ckpt = find_latest_checkpoint(config.checkpoint_dir)
# if ckpt:
#     meta = load_checkpoint(ckpt, model, logger=logger)
#     trainer.global_step  = meta['step']
#     config.resume_patch  = meta['patch_idx'] + 1
#     print(f'▶️  Resumed từ patch {meta["patch_idx"]}, step {meta["step"]}')

print('🚀 Bắt đầu training...')
final_metrics = trainer.train(train_samples, val_samples)

print(f'\n🎉 Training hoàn tất!')
print(f'   Best val loss : {trainer.best_val_loss:.4f}')
print(f'   Final val MCD : {final_metrics.get("val_mcd", "N/A")}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 12 — Inference: tổng hợp giọng nói từ text
# ════════════════════════════════════════════════════════════
from xtts_finetune.inference import run_inference

test_texts = [
    'Xin chào, đây là hệ thống chuyển văn bản thành giọng nói tiếng Việt.',
    'Hôm nay trời đẹp, tôi rất vui được gặp bạn.',
    'Công nghệ trí tuệ nhân tạo đang phát triển rất nhanh.',
]

for i, text in enumerate(test_texts):
    out_path = f'{OUTPUT_DIR}/test_{i+1:02d}.wav'
    run_inference(
        text            = text,
        reference_audio = config.reference_audio,
        output_path     = out_path,
        config          = config,
        logger          = logger,
    )
    print(f'✅ [{i+1}] {out_path}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 13 — Nghe thử audio ngay trong notebook
# ════════════════════════════════════════════════════════════
from IPython.display import Audio, display

for i, text in enumerate(test_texts):
    path = f'{OUTPUT_DIR}/test_{i+1:02d}.wav'
    if os.path.isfile(path):
        print(f'\n📢 [{i+1}] "{text[:70]}"')
        display(Audio(path))

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 14 — Zip output để download
# ════════════════════════════════════════════════════════════
import shutil

zip_path = '/kaggle/working/xtts_output.zip'
shutil.make_archive('/kaggle/working/xtts_output', 'zip', OUTPUT_DIR)

size_mb = os.path.getsize(zip_path) / 1024**2
print(f'✅ Đã zip: {zip_path} ({size_mb:.1f} MB)')
print('👉 Download: Kaggle → Output tab → xtts_output.zip')